# Prepare Calendar Table

Creates `adb_caresync_smd.silver.calendar` — a standard date dimension from 1940-01-01 to 2030-12-31.

In [0]:
CREATE OR REPLACE TABLE adb_caresync_smd.silver.calendar AS

WITH dates AS (
  SELECT explode(sequence(DATE'1940-01-01', DATE'2030-12-31', INTERVAL 1 DAY)) AS date
)

SELECT
  -- Keys & core date
  CAST(date_format(date, 'yyyyMMdd') AS INT)       AS date_key,
  date                                              AS date,

  -- Year
  year(date)                                        AS year,
  CASE WHEN (year(date) % 4 = 0 AND year(date) % 100 != 0)
            OR year(date) % 400 = 0
       THEN true ELSE false END                     AS is_leap_year,

  -- Quarter
  quarter(date)                                     AS quarter,
  CONCAT('Q', quarter(date))                        AS quarter_name,
  CONCAT(year(date), '-Q', quarter(date))            AS year_quarter,

  -- Month
  month(date)                                       AS month,
  date_format(date, 'MMMM')                         AS month_name,
  date_format(date, 'MMM')                           AS month_name_short,
  CONCAT(year(date), '-', LPAD(month(date), 2, '0')) AS year_month,

  -- Week
  weekofyear(date)                                  AS week_of_year,
  CONCAT(year(date), '-W', LPAD(weekofyear(date), 2, '0')) AS year_week,

  -- Day
  day(date)                                         AS day_of_month,
  dayofyear(date)                                   AS day_of_year,
  dayofweek(date)                                   AS day_of_week,
  date_format(date, 'EEEE')                         AS day_name,
  date_format(date, 'EEE')                           AS day_name_short,

  -- Weekend / Weekday
  CASE WHEN dayofweek(date) IN (1, 7) THEN true ELSE false END  AS is_weekend,
  CASE WHEN dayofweek(date) NOT IN (1, 7) THEN true ELSE false END AS is_weekday,

  -- Period boundaries – Month
  trunc(date, 'MM')                                 AS first_day_of_month,
  last_day(date)                                    AS last_day_of_month,
  datediff(date, trunc(date, 'MM')) + 1             AS day_of_month_ordinal,
  datediff(last_day(date), date) + 1                AS days_remaining_in_month,
  day(last_day(date))                               AS days_in_month,

  -- Period boundaries – Quarter
  trunc(date, 'quarter')                            AS first_day_of_quarter,
  date_sub(add_months(trunc(date, 'quarter'), 3), 1) AS last_day_of_quarter,

  -- Period boundaries – Year
  trunc(date, 'year')                               AS first_day_of_year,
  DATE(CONCAT(year(date), '-12-31'))                 AS last_day_of_year,

  -- Period boundaries – Week (Monday start)
  date_sub(date, (dayofweek(date) + 5) % 7)         AS first_day_of_week,
  date_add(date_sub(date, (dayofweek(date) + 5) % 7), 6) AS last_day_of_week,

  -- Relative flags (useful for reporting)
  CASE WHEN date = current_date() THEN true ELSE false END           AS is_today,
  CASE WHEN year(date) = year(current_date())
        AND month(date) = month(current_date()) THEN true ELSE false END AS is_current_month,
  CASE WHEN year(date) = year(current_date())
        AND quarter(date) = quarter(current_date()) THEN true ELSE false END AS is_current_quarter,
  CASE WHEN year(date) = year(current_date()) THEN true ELSE false END AS is_current_year

FROM dates
ORDER BY date;

In [0]:
SELECT count(*) AS total_rows,
       min(date) AS min_date,
       max(date) AS max_date
FROM   adb_caresync_smd.silver.calendar;
